# MapleStory Job Category Classifier
This notebook demonstrates a retrieval-augmented LLM workflow.

The notebook:

Retrieves MapleStory class information from the DigitalTQ website.
Extracts the website text content.
Sends the content to GPT-4.1-mini.
Uses the LLM to determine which category contains a specified MapleStory job.
Restricts the model to using only the retrieved website content.
Example query:

Angelic Buster → Nova
Kinesis → Other (if present in the retrieved text)
The notebook also includes basic validation checks to verify whether the target job appears in the scraped website content before classification.

## Imports and OpenAI Client

In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

client = OpenAI()

## Helper Function

In [ ]:
def fetch_website_contents(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    return soup.get_text("\n", strip=True)

## Download Website Text

In [ ]:
url = "https://www.digitaltq.com/maplestory-classes-jobs"

maplestory = fetch_website_contents(url)

print(f"Website text length: {len(maplestory):,}")

## Configure the Question

In [ ]:
job_name = "Kain"

print(f"Searching for: {job_name}")
print(f"Found in scraped text? {job_name in maplestory}")
print(f"Position: {maplestory.find(job_name)}")

## Build the Prompts

In [ ]:
system_prompt = """
You are a MapleStory classifier.

Use ONLY the website text provided.

Rules:
1. Find the exact job name.
2. Quote the exact text that contains the job name.
3. Identify the nearest category heading above that job.
4. If the exact job name does not appear in the website text, return UNKNOWN.
5. Do not use prior MapleStory knowledge.
6. Do not assume missing content exists.

Return:

Evidence:
...

Category:
...
"""

In [ ]:
user_prompt = f"""
Website information:

{maplestory}

Question:

Which category contains {job_name}?

First explain how you found it.
Then give the category.
"""

## Create Messages

In [ ]:
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

## Call the LLM

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    temperature=0
)

print(response.choices[0].message.content)